# Ollama Multiagent ( interest, planner, specialist , custom agent delegator and summarized agent) cat language V1
## sources:
https://google.github.io/adk-docs/agents/llm-agents/#equipping-the-agent-tools-tools<br>
https://google.github.io/adk-docs/agents/custom-agents/<br>
https://google.github.io/adk-docs/agents/multi-agents/<br>


In [2]:
%%capture 
#omet la sortida
'''
#llista de llibreries que s'han d'afegir a requeriments.in
#!{sys.executable} -m pip install litellm
!{sys.executable} -m pip install git+https://github.com/google/adk-python.git@main
#!{sys.executable} -m pip install nbconvert[webpdf]
#!{sys.executable} -m pip install -U ddgs
'''

In [3]:
%%capture
# llançament d'una aplicacio en l'entorn virtual
!{sys.executable} adk web --port 9000 "C:\Users\fijo\Documents\PYTHON\sample-agent"

In [4]:
%%capture
'''
# Esbrinar contingut d'una llibreria
import inspect
import google.adk.events.event_actions

# Obtenim tots els "membres" del mòdul adk
# members --> class, method, function, generator, traceback, frame, code, 
members = inspect.getmembers(google.adk.events.event_actions)

# Imprimim només els que són classes
for name, member_class in members:
    if inspect.ismodule(member_class):
        print(f"Modul trobat: {name}")
    elif inspect.isclass(member_class):
        print(f"Classe trobada: {name}")
    elif inspect.isfunction(member_class):
        print(f"funcio trobada: {name}")
'''       

In [5]:
%%capture
'''
import google.adk.events.event_actions
import inspect
import pkgutil

for m in pkgutil.iter_modules(google.adk.events.__path__):
    print("Submòdul:", m.name)


def buscar(obj, texto):
    for name, member in inspect.getmembers(obj):
        if texto.lower() in name.lower():
            print(name, member)

buscar(google.adk, "step")
'''

In [6]:
# carrega lliberies base
import os
import sys
import re
import traceback
import inspect
import setuptools

In [7]:
%%capture
#lança ollama
import subprocess
subprocess.Popen('start "" "C:\\Users\\fijo\\AppData\Local\Programs\Ollama\\ollama.exe" list', shell=True)

In [8]:
#Carrega mes llibreries
import litellm
litellm.api_base = "http://localhost:11434"
# Desactivem els callbacks d'èxit i error perquè no intenti fer feina en segon pla
litellm.success_callback = []
litellm.failure_callback = []
# Opcional: Desactivar telemetria per guanyar una mica de velocitat
litellm.telemetry = False
from typing import AsyncGenerator
# google-adk instalat desde  git+https://github.com/google/adk-python.git@main
from google.adk.agents import Agent, LlmAgent, BaseAgent, SequentialAgent
from google.adk.agents.invocation_context import InvocationContext
from google.adk.events import Event
from google.adk.runners import InMemoryRunner
from google.genai import types
from google.genai.types import Content, Part
from google.adk.tools import AgentTool, FunctionTool
import warnings
#from ddgs import DDGS

In [9]:
%%capture
# Seleccionem el model
#local_model = "ollama/gemma3:1b"
local_model = "ollama/gemma3n:e4b"
#local_model = "ollama/qwen3-vl:4b"
#local_model = "qwen3:8b"
#local_model = "mistral:7b"
#local_model = "ollama/mistral-7B-Instruct-v0.2"

# declarem els agents

# Aquest agent inicia la conversa per entendre l'usuari i refinar el objectiu.
interest_analysis_agent = LlmAgent(
    model=local_model,
    name="InterestAnalysisAgent",
    description="Analitza la necessitat de l'usuari fent preguntes per clarificar l'objectiu de la seva pregunta inicial",
    instruction="""La teva tasca és ajudar a aclarir la pregunta inicial de l’usuari fins a obtenir
    un OBJECTIU DE RECERCA clar, concret i útil per al següent agent del sistema.
    NORMES:
    
    1. NO RESPONGUIS la pregunta de fons en cap moment.
    2. En cada torn, si encara no tens prou informació, formula UNA ÚNICA
       pregunta aclaridora, breu i directa.
    3. NO POTS fer servir ###STOP### si la teva sortida és una pregunta.
    4. NO POTS fer servir ###STOP### fins que:
       - l’usuari hagi respost almenys una vegada,
       - i hagis obtingut prou informació per definir un objectiu clar.
    5. Quan ja tens prou informació per definir l’objectiu, NO pots fer cap més pregunta.
       En aquest cas:
       - genera ÚNICAMENT una frase declarativa que expressi l’OBJECTIU DE RECERCA,
       - i acaba amb ###STOP###,
       - sense cap pregunta, sense cap explicació i sense cap text addicional.
    6. La sortida final ha de ser sempre un OBJECTIU DE RECERCA en forma declarativa.
       Mai pot ser una pregunta.
    7. NO repeteixis ###STOP### en torns successius.
    8. NO facis preguntes circulars, vagues o autoreferencials.
    9. NO demanis a l’usuari “què creu ell”; dedueix tu mateix la intenció a partir
       de les seves respostes.
    10. Si l’usuari indica explícitament que la clarificació és suficient
        (“ja està bé”, “endavant”, “continua”, “ja ho tens”, “pots passar al següent pas”),
        NO facis cap més pregunta.
        En aquest cas:
        - resumeix la intenció de l’usuari en UNA sola frase clara,
        - converteix-la en un OBJECTIU DE RECERCA,
        - i acaba amb ###STOP###.
    11. Si l’usuari respon amb “sí”, “correcte”, “exacte”, “això mateix”
        o qualsevol confirmació equivalent, i ja tens prou informació,
        NO facis cap més pregunta.
        En aquest cas:
        - genera directament l’OBJECTIU DE RECERCA,
        - i acaba amb ###STOP###.
    La teva sortida és sempre:
    - o bé una pregunta aclaridora,
    - o bé un OBJECTIU DE RECERCA + ###STOP###
    """
    )

# Aquest agent emet el pla una vegada clarificat el objectiu de cerca
planner_agent = LlmAgent(
    model = local_model,
    name = "PlannerAgent",
    description = " Analitzar la questio entregada pel InterestAnalysisAgent \
    i crea un pla per respondra-la",
    instruction = """ Rebreu un OBJECTIU DE RECERCA definit per l'analista.
    La vostra tasca és crear un pla de 2 a 4 passos utilitzant EXCLUSIVAMENT els agents següents:
        - HistoryAgent
        - GeographyAgent
        - SociologyAgent
    NORMES CRITIQUES:
    1. La teva sortida ha de ser UNICAMENT el pla, sense cap text introductori ni explicacio.
    2. Has de fer servir exactament aquest fromat per cada linia:
        [Numero de pas]. [Nom de l'agent], [Descripcio detallada de la tasca]

    EXEMPLE DE SORTIDA:
    1. HistoryAgent, Investigar el context polític de Hawaii el 2002.
    2. GeographyAgent, Descriure l'illa de Kauai i la seva orografia.

    Tota la teva resposta es desara automaticament a la clau 'plan' de l'estat.
    """,
    output_key = "plan",
    generate_content_config = types.GenerateContentConfig(temperature = 0.0)
    )

#Agents especialistes 
geography_agent = LlmAgent(   
    model=local_model,
    name="GeographyAgent",
    description="Respon a preguntes sobre geografia relacionades \
    amb ubicacions i terreny",
    instruction=""" Ets un expert en geografia.
    -La teva tasca és: {current_task_instruction}.
    -Respon a la pregunta de l'usuari de manera concisa i factual basant-te 
    en dades geogràfiques.
    -Proporciona NOMÉS la resposta, sense cap mena de conversa.
    """,
    output_key = "geography_result",
    generate_content_config =types.GenerateContentConfig(temperature = 0.7),
    #sub_agents = [],
    #tools=[custom_web_search()] 
    # tools will be added next
    )

history_agent = LlmAgent(   
    model=local_model,
    name="HistoryAgent",
    description= "Respon a preguntes sobre història pel que fa al \
    context polític i les dates." ,
    instruction=""" Ets un expert en història.
    -La teva tasca és: {current_task_instruction}.
    -Respon a la pregunta de l'usuari de manera concisa i factual, 
        basant-te en dades històriques.
    -Proporciona NOMÉS la resposta, sense cap mena de conversa.
    """,
    output_key = "history_result",
    generate_content_config = types.GenerateContentConfig(temperature = 0.7),
    #sub_agents = [],
    #tools=[custom_web_search()] 
    # tools will be added next
    )

sociology_agent = LlmAgent(   
    model=local_model,
    name="SociologyAgent",
    description="Respon a preguntes sobre sociologia pel que fa a \
    l'impacte social i els moviments.",
    instruction=""" Ets un expert en sociologia.
    -La teva tasca és: {current_task_instruction}
    -Respon a la pregunta de l'usuari de manera concisa i factual basant-te 
        en dades sociològiques.
    -Proporciona NOMÉS la resposta, sense cap mena de conversa.
    """,
    output_key = "sociology_result",
    generate_content_config =types.GenerateContentConfig(temperature = 0.7),
    #sub_agents = [],
    #tools=[custom_web_search()] 
    # tools will be added next
    )


In [10]:
# Declaracio del DynamicCoordinator que delega les parts del pla a cada especialista
class DynamicCoordinatorAgent(BaseAgent):
    """
    Coordinador programat que executa el pla de la sessio
    cridant als agents especialistes un per un
    """
    # Declaracio de tipus per a Pydantic
    history_agent: LlmAgent
    geography_agent: LlmAgent
    sociology_agent: LlmAgent
    synthesizer_agent: LlmAgent
    #afegim el experts_map com a tipus dict
    experts_map: dict ={}

    model_config = {"arbitrary_types_allowed": True}

    def __init__(self, name, history_agent, geography_agent, sociology_agent, synthesizer_agent):
        # Llista d'agents per a la jerarquia de l'ADK
        sub_agents_list = [history_agent, geography_agent, sociology_agent, synthesizer_agent]

        # Mapa per traduir el nom del pla a l'instància de l'agent [8]
        experts_dict = {
            "HistoryAgent": history_agent,
            "GeographyAgent": geography_agent,
            "SociologyAgent": sociology_agent
            }
        #Es pasen tots els objectes al constructor del pare y Pydantic els 
        #assigna al self
        super().__init__(
            name = name, 
            sub_agents = sub_agents_list,      
            history_agent = history_agent,
            geography_agent = geography_agent,
            sociology_agent = sociology_agent,
            synthesizer_agent = synthesizer_agent,
            experts_map = experts_dict
            )


    
    async def _run_async_impl(self, ctx: InvocationContext) -> AsyncGenerator[Event, None]:
        # 1. Obtenir el pla desat pel PlannerAgent
        pla_text = ctx.session.state.get("plan", "")
        if not pla_text:
            yield Event(author=self.name, content="ERROR: No s'ha trobat cap pla a la sessió.")
            return

        # 2. Parsejar el pla (ex: "1. HistoryAgent, Descripció...")
        # El regex busca: número, nom de l'agent i descripció
        tasks = re.findall(r'(\d+)\.\s*(\w+),\s*(.*)', pla_text)
        
        # 3. Execució seqüencial dels especialistes
        for step_id, agent_name, task_desc in tasks:
            agent = self.experts_map.get(agent_name.strip())
            if agent:
                # Podem passar la tasca específica a través d'una clau temporal a l'estat 
                ctx.session.state["current_task_instruction"] = task_desc
                
                # Cridem l'especialista i fem 'yield' dels seus esdeveniments
                async for event in agent.run_async(ctx):
                    yield event
            else:
                yield Event(author=self.name, content=f"AVÍS: Agent {agent_name} no reconegut.")

        # 4. Finalment, cridem al SynthesizerAgent per tancar el report 
        async for event in self.synthesizer_agent.run_async(ctx):
            yield event  
    

In [11]:
# Declaracio del Synthesizer Agent  
synthesizer_agent = LlmAgent(
    model = local_model,
    name = "SynthesizerAgent",
    description = "Sintetitzar la respost final al usuari en catala",
    instruction="""Ets un expert sintetitzador d'informació. 
    Has de generar un informe final coherent i ben estructurat en catala \
    basant-te en les següents troballes dels experts:
    
    #El ? evita que l'agent doni un error si la variable no s'ha declarat al pla
    HISTÒRIA: {history_result?}
    GEOGRAFIA: {geography_result?}
    SOCIOLOGIA: {sociology_result?}

    OBJECTIU:
    - Crea una narrativa fluida que respongui a la consulta inicial de l'usuari.
    - No mencionis els noms dels agents (com 'HistoryAgent'), presenta la informació de forma professional.
    - Si alguna de les seccions anteriors està buida o no s'ha trobat informació, omet-la de l'informe.
    - Assegura't que el to sigui informatiu i educat.
    """,
    #tools = [read_tool],  
    output_key = "final_result",
    generate_content_config= types.GenerateContentConfig(temperature = 0.7),
    )


In [12]:
%%capture
# Ignorar només els RuntimeWarning i el warning de pydantic 
# (per diferencies de versions entre pydantic  i ollama , s'espera 10 camps i troba 6)
warnings.filterwarnings("ignore", category = RuntimeWarning)
warnings.filterwarnings("ignore", module = "pydantic")

# Hi han dos pipelines.1º interve el interes_analysis_agent afegint refinament \
# a la pregunta inicial
# el segon pipeline es per arrencar el planner agent i resta d'agents
# es comuniquen a traves de l'estat compartit de la sessio (session.state)
clarification_pipeline = SequentialAgent(
    name = "ClarificationPipeline",
    sub_agents = [interest_analysis_agent]
    )
# 2. Instanciar els Runners a cada pipeline per a cada sequential agent
clarification_runner = InMemoryRunner(agent = clarification_pipeline)
# Els runners necessiten inicialitzar l'espai de memòria per a aquesta ID.
clarification_runner.auto_create_session = True

#Declarem al custom coordinator
#es declaren tots els agents especialistes + el synthesizer com a sub-agents
dynamic_coordinator = DynamicCoordinatorAgent(
    name = "Coordinator",
    history_agent = history_agent,
    geography_agent = geography_agent,
    sociology_agent = sociology_agent,
    synthesizer_agent = synthesizer_agent
    )
# Es defineix la pipeline d'execucio
final_pipeline = SequentialAgent(
    name = "ExecutionPipeline",
    sub_agents = [
        planner_agent,      #Genera el pla i el guarda a session.state["plan"]
        dynamic_coordinator # llegeix el pla de l'estat i el passa als especialistes
        ]
    )
# Es defineix el runner que crinda a la final_pipeline
execution_runner = InMemoryRunner(agent = final_pipeline)
execution_runner.auto_create_session = True

In [13]:
#Execucio de la Sessio
# Definicio de la funcio d'entrada inicial
def pregunta_en_dues_fases(pregunta_inicial, session_id=None, user_id = "usuari"):

    print(f"🌍 question :'{pregunta_inicial}'..")
    # Gestio de la sessio
    if not session_id:
        session_id="user_default"
        print(f"Nova sessio creada amb ID: {session_id}")
    

    clarify_session_id =f"{session_id}_clarify"

    # --- FASE 1: BUCLE DE CLARIFICACIO PREGUNTA USUARI ---
    print("\n--- Fase 1: Analitzant i aclarint la pregunta inicial---")
    pregunta_clarificada = ""
    missatge_actual = Content(role = "user", 
                              parts = [Part(text = pregunta_inicial)])
    # Limitem el nombre de torns per seguretat
    MAX_TURNS = 5
    for i in range(MAX_TURNS):
        print(f"\n--- torn de clarificacio{i+1} ---")
        try:
            clarification_responses = clarification_runner.run(
                new_message = missatge_actual,
                session_id = clarify_session_id,
                user_id = user_id
                )

            resposta_agent = ""
            for event in clarification_responses:
                #---------------------------------
                #print("TIPUS:", type(event))
                #print(event)
                #print("----")
                #---------------------------------
                if(hasattr(event, 'author') and 
                   event.author == "InterestAnalysisAgent" and
                   event.is_final_response()):
                    if event.content and event.content.parts:
                        resposta_agent = event.content.parts[0].text
                        break
            print(f"Agent analista conclou: {resposta_agent}")
            marca_final = "###STOP###"

            # Es comprova si l'agent ha donat la clarificacio per acabada
            if marca_final in  resposta_agent:
                # Es neteja la resposta final per deixar no mes l'objectiu
                pregunta_clarificada = resposta_agent.replace(marca_final, " ").strip()
                print(f"\n Objectiu clarificat: '{pregunta_clarificada}'")
                break # sortida del bucle
            #si no ha acabat, es torna a preguntar a l'usuari
            resposta_usuari = input("La teva resposta: ")
            missatge_actual = Content(role = "user", parts = [Part(text = resposta_usuari)])
        except Exception as e:
            print(f"\n S'ha produit un error en la fase d'aclariments: {e}")
            return #s'atura l'execucio
    # Si despres de MAX_TURNS no s'ha aclarit, s'atur el proces
    if not pregunta_clarificada:
        print(f"\n No s'ha pogut acabar la fase de clarificacio. S'atura el proces.")
        return #Fi de Fase 1


    # --- FASE 2 : EXECUCIO DEL PLA ---
    print("\n --- Fase 2: creant un pla enfocat i executant---")
    try:
        #Combinacio de la pregunta inicial amb la aclariment final per donar contexte al planner
        context_enriquit = f"Pregunta inicial de l'usuari: '{pregunta_inicial}'. \
                            \n Objectiu final clarificat: '{pregunta_clarificada}'"
        missatge_enriquit = Content(role = "user", parts = [Part(text = context_enriquit)])

        # Llançament del execution_runner 
        # el sequential agent llança planner-->dynamiccoordinator-->synthesizer
        execution_responses = execution_runner.run( 
                                new_message = missatge_enriquit,
                                session_id = f"{session_id}_dynamic", 
                                user_id = user_id
                                )
        # Variable de control per aturar el bucle si detectem el final
        pla_finalitzat = False
        for event in execution_responses:
            #----------------------------
            #print("TIPUS:", type(event))
            #print(event)
            #print("-"*60)
            #----------------------------
            if event.content and event.content.parts:
                #print(f"[{event.author}]:{event.content.parts[0].text}")
                if event.author == "SynthesizerAgent":
                    print("-"*60)
                    print("\n --- Resposta final ---")
                    print(f"[{event.author}]:{event.content.parts[0].text}")
    except Exception as e:
        print(f"\n S'ha produit un error en la 2º fase , execucio del pla: {e}")
        traceback.print_exc() # Això t'ajudarà a veure on falla exactament


In [14]:
la_sessio="chat1"
pregunta_en_dues_fases("que em pots dir del personatje de Fausto en l'obra de Goethe?", session_id=la_sessio, user_id="usuari")

🌍 question :'que em pots dir del personatje de Fausto en l'obra de Goethe?'..

--- Fase 1: Analitzant i aclarint la pregunta inicial---

--- torn de clarificacio1 ---
Agent analista conclou: Pots especificar quin aspecte del personatge de Fausto t'interessa més? Per exemple, la seva motivació, la seva transformació, les seves relacions amb altres personatges?



La teva resposta:  M'interessen totes les facetes. la pregunta ja esta clarificada. endavant



--- torn de clarificacio2 ---
Agent analista conclou: El personaatge de Fausto i les seves múltiples facetes ###STOP###

 Objectiu clarificat: 'El personaatge de Fausto i les seves múltiples facetes'

 --- Fase 2: creant un pla enfocat i executant---
------------------------------------------------------------

 --- Resposta final ---
[SynthesizerAgent]:## El Personatge de Faust i les seves Múltiples Facetes

Faust, personatge central de l'obra homònima de Goethe, és una figura complexa i ricament interpretada al llarg de la història. La seva representació ha evolucionat significativament, reflectint els canvis en les perspectives culturals, psicològiques i socials.

Inicialment, durant l'època de Goethe (principis del segle XIX), Faust va ser considerat una encarnació de l'individu romàntic. Aquesta interpretació destaca la seva impulsivitat, la seva recerca d'experiències transcendents i el seu conflicte amb les normes morals i el saber acadèmic. Faust personifica la rebel·lió contr

In [ ]:
la_sessio="chat1"
pregunta_en_dues_fases("Es pot fer un simil de Faust a la actualitat amb els gobernants i el poder que poden assolir amb eines de IA", session_id=la_sessio, user_id="usuari")